# Training an LLM Specifically for Proof of Thought

In [4]:
#!/usr/bin/env python
# coding: utf-8

# Advanced RAG on Hugging Face documentation using LangChain (with some packages removed because I only need the generation part of RAG)
# taken from https://huggingface.co/learn/cookbook/en/advanced_rag
# and also from https://python.langchain.com/v0.2/docs/integrations/document_loaders/url/
# and also from the benchmark/benchmark_full.ipynb

import json
import logging
import sys
import os
from pathlib import Path
from typing import Literal, Any, Optional, List, Tuple
from transformers import AutoTokenizer
import huggingface_hub
import transformers
import torch

# Basically the IPython Magic version of these Linux commands for installations
# cd ..
# pip install -r requirements.txt
# pip install -e .
path_words = os.getcwd().split("/")
if path_words[len(path_words)-1] == "benchmark":
    %cd ..
%pip install -r requirements.txt
%pip install -e .

# Add parent directory to path for z3adapter imports
sys.path.insert(0, str(Path(os.getcwd()).parent.parent))

from utils.azure_config import get_client_config
from z3adapter.reasoning import EvaluationPipeline, ProofOfThought
from openai import OpenAI

# Configure logging (output messages during the benchmark). This stores all logs in the outputLog.txt file.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("outputLog.txt", mode="a"),
        logging.StreamHandler(),
    ],
)

print('done importing')

Note: you may need to restart the kernel to use updated packages.
Obtaining file:///home/kyi/proofofthought
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for proofofthought (pyproject.toml) ... done
  Created wheel for proofofthought: filename=proofofthought-1.0.1-0.editable-py3-none-any.whl size=8343 sha256=9e3362481a950e6006a9cc09932127f967d0b1d3006348f5888163d83aeb735d
  Stored in directory: /scratch/kyi/job_51787006/pip-ephem-wheel-cache-8jr1f_0j/wheels/b1/a5/e7/03f593fd45d9567f8de68be4e81027f53a9713de5b5bce5d39
Successfully built proofofthought
  Attempting uninstall: proofofthought
    Found existing installation: proofofthought 1.0.1
    Uninstalling proofofthought-1.0.1:
      Successfully uninstalled proofofthought-1.0.1
Note: you may need to restart the kernel to use updated packages.
done importi

In [7]:
! hf auth login --token blank  #Your Hug Face authorization token

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `proof-of-thought-llm-training` has been saved to /home/kyi/.cache/huggingface/stored_tokens
Your token has been saved to /home/kyi/.cache/huggingface/token
Login successful.
The current active token is: `proof-of-thought-llm-training`


In [9]:
#Set up model and tokenizer
model = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model)
print(tokenizer)

2026-07-06 12:02:24,065 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-06 12:02:24,175 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"
2026-07-06 12:02:24,440 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

2026-07-06 12:02:24,681 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-06 12:02:24,796 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-06 12:02:24,974 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

2026-07-06 12:02:25,145 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-07-06 12:02:25,236 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-07-06 12:02:25,323 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
2026-07-06 12:02:25,431 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/vocab.json "HTTP/1.1 200 OK"
2026-07-06 12:02:25,592 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/vocab.json "HTTP/1.1 200 OK"


...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

2026-07-06 12:02:27,041 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/tokenizer.json "HTTP/1.1 307 Temporary Redirect"
2026-07-06 12:02:27,151 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/tokenizer.json "HTTP/1.1 200 OK"
2026-07-06 12:02:27,328 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/tokenizer.json "HTTP/1.1 200 OK"


tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

2026-07-06 12:02:27,824 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
2026-07-06 12:02:27,976 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/special_tokens_map.json "HTTP/1.1 404 Not Found"
2026-07-06 12:02:28,146 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-07-06 12:02:28,623 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct "HTTP/1.1 200 OK"


Qwen2Tokenizer(name_or_path='Qwen/Qwen2.5-1.5B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstrip=Fa